# Analytics Agent - CLI Notebook

A **Text-to-SQL ReAct Agent** that translates natural language questions into SQL queries.

**Supported LLMs:** Groq, Azure OpenAI, Perplexity

# Cell 1: Define your envs

In [ ]:
LLM_CONFIG={}

LANGFUSE_SECRET_KEY = "sk-example-12345"
LANGFUSE_PUBLIC_KEY = "pk-example-3214123"
LANGFUSE_BASE_URL = "https://cloud.langfuse.com"


GROQ_API_KEY="gsk_example"

PORT=3050

POSTGRES_DB="example"
POSTGRES_HOST = "localhost"
POSTGRES_USER = "example"
POSTGRES_PASSWORD = "example"
POSTGRES_PORT= 5432

## Cell 2: Environment Variables

In [16]:
import os

# Groq API Key (required)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# PostgreSQL Connection Settings
os.environ["POSTGRES_HOST"] = POSTGRES_HOST
os.environ["POSTGRES_PORT"] = str(POSTGRES_PORT)
os.environ["POSTGRES_DB"] = POSTGRES_DB
os.environ["POSTGRES_USER"] = POSTGRES_USER
os.environ["POSTGRES_PASSWORD"] = POSTGRES_PASSWORD

# Langfuse (optional - for observability)
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_ENVIRONMENT"] = "development"
os.environ["LANGFUSE_BASE_URL"] = LANGFUSE_BASE_URL

## Cell 3: Standard Library Imports

In [17]:
import asyncio
import uuid
import json
import threading
from abc import ABC, abstractmethod
from functools import lru_cache

## Cell 4: Langfuse Setup

In [18]:
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler


langfuse_instance = Langfuse(
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
    environment=os.environ.get("LANGFUSE_ENVIRONMENT", ""),
    base_url=os.environ.get("LANGFUSE_BASE_URL", ""),
)

langfuse_callback = CallbackHandler()

## Cell 5: Tools Abstract Class

In [19]:
import pandas as pd
from langchain_core.tools import StructuredTool


class Tools(ABC):
    @property
    @abstractmethod
    def tools(self) -> list[StructuredTool]:
        ...

    @property
    @abstractmethod
    def tool_get_all_table(self) -> StructuredTool:
        ...

    @property
    @abstractmethod
    def tool_get_table_schema(self) -> StructuredTool:
        ...

    @property
    @abstractmethod
    def tool_execute_query(self) -> StructuredTool:
        ...

    @abstractmethod
    def execute(self, query: str) -> list:
        ...

    @abstractmethod
    def execute_query(self, query: str) -> str:
        ...

    @abstractmethod
    def execute_df(self, query: str) -> pd.DataFrame:
        ...

    @abstractmethod
    def get_all_table(self) -> str:
        ...

    @abstractmethod
    def get_table_schema(self, table_name: str) -> str:
        ...

## Cell 6: PostgresTool Class

In [20]:
import psycopg2

from pydantic import BaseModel, Field


class ExecuteQueryInput(BaseModel):
    query: str = Field(..., description="The SQL query to be executed.")


class GetTableSchemaInput(BaseModel):
    table_name: str = Field(..., description="The name of the table.")


class EmptyInput(BaseModel):
    pass


class PostgresTool(Tools):
    def __init__(self, connection_params: dict):
        self._connection = None
        self.connection_params = connection_params
        self.db_lock = threading.Lock()

    @property
    def connection(self):
        if self._connection is None:
            self._connection = psycopg2.connect(**self.connection_params)
            self._connection.autocommit = True
        return self._connection

    def execute(self, query: str):
        with self.db_lock:
            with self.connection.cursor() as cur:
                cur.execute(query)
                if cur.description:
                    return cur.fetchall()
                return []

    def execute_query(self, query: str) -> str:
        try:
            print("[*] Executing query:", query)
            data = self.execute(query)
            return json.dumps(data[:100], default=str)
        except Exception as e:
            print("[!] Error:", e)
            return str(e)

    def execute_df(self, query: str) -> pd.DataFrame:
        with self.db_lock:
            return pd.read_sql_query(query, self.connection)

    def get_all_table(self, *args, **kwargs) -> str:
        query = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
        """
        tables = self.execute(query)
        return ",".join(t[0] for t in tables)

    def get_table_schema(self, table_name: str, *args, **kwargs) -> str:
        query = f"""
        SELECT
            column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_schema = 'public'
          AND table_name = '{table_name}'
        ORDER BY ordinal_position;
        """
        df = self.execute_df(query)
        if df.empty:
            raise Exception(f"Table '{table_name}' does not exist.")
        return self._df_to_create_statement(df, table_name)

    @staticmethod
    def _df_to_create_statement(df: pd.DataFrame, table_name: str) -> str:
        columns = []
        for _, row in df.iterrows():
            col = f"    {row['column_name']} {row['data_type']}"
            if row["is_nullable"] == "NO":
                col += " NOT NULL"
            columns.append(col)
        return f"CREATE TABLE {table_name} (\n" + ",\n".join(columns) + "\n);"

    @property
    def tool_get_all_table(self):
        return StructuredTool(
            name="get_all_tables",
            description="Retrieve all table names.",
            func=self.get_all_table,
            args_schema=EmptyInput,
        )

    @property
    def tool_get_table_schema(self):
        return StructuredTool(
            name="get_table_create_statement",
            description="Get CREATE TABLE statement.",
            func=self.get_table_schema,
            args_schema=GetTableSchemaInput,
        )

    @property
    def tool_execute_query(self):
        return StructuredTool(
            name="execute_query",
            description="Execute SQL queries.",
            func=self.execute_query,
            args_schema=ExecuteQueryInput,
        )

    @property
    def tools(self):
        return [
            self.tool_execute_query,
            self.tool_get_all_table,
            self.tool_get_table_schema,
        ]

## Cell 7: Initialize db_tool

In [21]:
@lru_cache
def get_postgres_tool() -> Tools:
    return PostgresTool(
        connection_params={
            "host": os.environ.get("POSTGRES_HOST", "localhost"),
            "port": int(os.environ.get("POSTGRES_PORT", "5432")),
            "dbname": os.environ.get("POSTGRES_DB", ""),
            "user": os.environ.get("POSTGRES_USER", ""),
            "password": os.environ.get("POSTGRES_PASSWORD", ""),
        }
    )


db_tool = get_postgres_tool()
print("Database tool initialized.")

Database tool initialized.


## Cell 8: StandaloneTextToSQLAgent Class

In [22]:
from langchain.agents import create_agent
from langchain.chat_models import BaseChatModel
from langchain_groq import ChatGroq
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver


class StandaloneTextToSQLAgent:
    PROMPT_TEMPLATE = """
You are an expert **Text-to-SQL ReAct Agent**. Translate natural language questions into SQL queries.

### Core Directives
1. Use `get_all_tables` and `get_table_create_statement` to retrieve schema when needed.
2. Write efficient SQL queries with `LIMIT 10` for large result sets.
3. Use `execute_query` to run queries.
4. Return only the data results, not the SQL.

### Tool Use (ReAct Pattern: Thought, Action, Observation)
- **get_all_tables**: Get all table names
- **get_table_create_statement**: Get table structure
- **execute_query**: Execute SQL queries

### Constraints
- Do not hallucinate table/column names
- Always execute queries when possible
"""

    def __init__(self, tools: list[StructuredTool], llm: BaseChatModel):
        self.tools = tools
        self.llm = llm
        self._agent = None

    @classmethod
    def from_azure_llm_config(cls, duckdb_tool: Tools, llm_config: dict, temperature: float):
        llm = AzureChatOpenAI(**llm_config, temperature=temperature)
        return cls.from_llm(duckdb_tool=duckdb_tool, llm=llm)

    @classmethod
    def from_perplexity(cls, duckdb_tool: Tools, api_key: str, temperature: float):
        llm = ChatOpenAI(
            api_key=api_key,
            base_url="https://api.perplexity.ai",
            model="sonar-reasoning-pro",
            temperature=temperature,
        )
        return cls.from_llm(duckdb_tool=duckdb_tool, llm=llm)

    @classmethod
    def from_groq(cls, duckdb_tool: Tools, api_key: str, temperature: float):
        llm = ChatGroq(api_key=api_key, temperature=temperature, model="qwen/qwen3-32b")
        return cls.from_llm(duckdb_tool=duckdb_tool, llm=llm)

    @classmethod
    def from_llm(cls, duckdb_tool: Tools, llm: BaseChatModel):
        if isinstance(duckdb_tool, Tools):
            duckdb_tool = duckdb_tool.tools
        return cls(duckdb_tool, llm)

    @property
    def agent(self):
        if self._agent is None:
            checkpointer = InMemorySaver()
            self._agent = create_agent(
                model=self.llm,
                tools=self.tools,
                system_prompt=self.PROMPT_TEMPLATE,
                checkpointer=checkpointer,
            )
        return self._agent

    def invoke(self, query: str):
        _input = {"messages": [{"role": "user", "content": query}]}
        agent_response = self.agent.invoke(input=_input)
        return agent_response["messages"][-1].content

## Cell 9: Initialize Agent

In [23]:
standalone_text_to_sql_agent = StandaloneTextToSQLAgent.from_groq(
    api_key=os.environ.get("GROQ_API_KEY", ""),
    duckdb_tool=db_tool,
    temperature=0,
)

agent = standalone_text_to_sql_agent.agent

print("Agent initialized with Groq (qwen/qwen3-32b)")

Agent initialized with Groq (qwen/qwen3-32b)


## Cell 10: run_in_thread Function

In [24]:
async def run_in_thread(question: str, thread_id: str):
    _input = {"messages": [{"role": "user", "content": question}]}
    result = await agent.ainvoke(
        _input,
        config={
            "configurable": {"thread_id": thread_id},
            "callbacks": [langfuse_callback],
        }
    )
    return result["messages"][-1].content

## Cell 11: run Function (CLI Loop)

In [25]:
async def run():
    thread_id = str(uuid.uuid4())
    
    while True:
        question = input("Enter your question: ")
        
        if question == "exit":
            break
        
        if question == "new":
            thread_id = str(uuid.uuid4())
            print("Started new conversation thread.")
            continue
        
        answer = await run_in_thread(question, thread_id)
        print("\nAnswer:", answer)

## Cell 12: Run Interactive CLI

In [28]:
# Start interactive CLI session
await run()

C:\Users\shtab\AppData\Local\Temp\ipykernel_18748\3437943521.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, self.connection)


[*] Executing query: SELECT DISTINCT product_name FROM dim_products WHERE category = 'phones' LIMIT 10;

Answer: There are no Samsung phones in the `dim_products` table. The query returned an empty result set for products categorized under 'phones'.


## Cell 13: Quick Test Query

In [27]:
# # Quick test
# result = asyncio.run(run_in_thread("What tables are available?", str(uuid.uuid4())))
# print("Answer:", result)